# Generation and post-analysis in a real dataset 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tableone import TableOne
import torch
from sksurv.nonparametric import kaplan_meier_estimator
import os
import sys
from pathlib import Path
module_path = Path.cwd().parent / 'utils'
sys.path.append(str(module_path))
import data_processing, visualization, metrics

module_path = Path.cwd().parent / 'execute'
sys.path.append(str(module_path))

from synthcity.metrics.plots import plot_tsne
from synthcity.metrics.eval import Metrics
from synthcity.metrics.scores import ScoreEvaluator
from synthcity.plugins.core.dataloader import SurvivalAnalysisDataLoader
from synthcity.utils.reproducibility import clear_cache, enable_reproducible_results

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Original data loading, analysis and visualization

### 1.1. Experiment setting - links to data

In [ ]:
dataset_name = "ACTG320"
current_path = os.getcwd()  # Get current working directory
parent_path = os.path.dirname(current_path)
data_file_control= parent_path + "/dataset/" + dataset_name + "/data_control.csv"
feat_types_file_control = parent_path + "/dataset/" + dataset_name + "/data_types_control.csv"
data_file_treated= parent_path + "/dataset/" + dataset_name + "/data_treated.csv"
feat_types_file_treated= parent_path + "/dataset/" + dataset_name + "/data_types_treated.csv"

# If the dataset has no missing data, leave the "miss_file" variable empty
miss_file = ""
true_miss_file = None

### 1.2. Loading, feature analysis, and visualization of original data: control vs treated group

In [ ]:
control_fnames = ['time', 'censor'] + pd.read_csv(feat_types_file_control)["name"].to_list()[1:]
control = pd.read_csv(data_file_control, header=None, names=control_fnames)
print(control.head())

In [ ]:
# Load and transform control data
data_init_control_encoded, feat_types_dict, miss_mask_control, true_miss_mask_control, _ = data_processing.read_data(data_file_control, 
                                                                                                             feat_types_file_control, 
                                                                                                             miss_file, true_miss_file)

In [ ]:
# Load and transform control data
df_init_control_encoded, feat_types_dict, miss_mask_control, true_miss_mask_control, _, dict_cat_map_control = data_processing.read_data(data_file_control, feat_types_file_control, miss_file, true_miss_file,
                                                                                                                                         return_cat_mapping=True)
data_init_control_encoded = torch.from_numpy(df_init_control_encoded.values)
data_init_control = data_processing.discrete_variables_transformation(data_init_control_encoded, feat_types_dict)

# Load and transform treated data
df_init_treated_encoded, _, _, _, _, dict_cat_map_treated = data_processing.read_data(data_file_treated, feat_types_file_treated, miss_file, true_miss_file,
                                                                                      return_cat_mapping=True)
data_init_treated_encoded = torch.from_numpy(df_init_treated_encoded.values)
data_init_treated = data_processing.discrete_variables_transformation(data_init_treated_encoded, feat_types_dict)

In [ ]:
# Format data in dataframe
df_init_treated = pd.DataFrame(data_init_treated.numpy(), columns=control_fnames)
df_init_control = pd.DataFrame(data_init_control.numpy(), columns=control_fnames)

# Update the data
df_init_treated["treatment"] = 1
df_init_control["treatment"] = 0
df_init = pd.concat([df_init_control, df_init_treated], ignore_index=True)

In [ ]:
continuous = [row['name'] for row in feat_types_dict if row['type'] in ['pos', 'real']]
categorical = [row['name'] for row in feat_types_dict if row['type'] in ['cat']]

## 2. Model training and data generation for different ε levels

In [ ]:
#We use the piecewise variant as a default.
for i in range(len(feat_types_dict)):
    if feat_types_dict[i]['name'] == "survcens":
        feat_types_dict[i]["type"] = 'surv_weibull'

### 2.1. Whole dataset

In [ ]:
import json
name_config_1 = f"best_params_traincontrol_{dataset_name}_aug_Ncontrol3%3"
name_config_2 = "ntrials150_DetectXGB"

generators_sel = ["HI-VAE_weibull", "HI-VAE_piecewise", "HI-VAE_weibull_DP", "HI-VAE_piecewise_DP"]

best_params_dict = {}
for generator_name in generators_sel:
    best_param_file = ""
    if "DP" not in generator_name:
        # best_param_file = [item for item in best_param_files if generator_name in item][0]
        for f in os.listdir(parent_path + "/dataset/" + dataset_name + "/optuna_results"):
            if (f.endswith(generator_name + '.json') & (name_config_1 in f) & (name_config_2 in f)):
                best_param_file = f
        with open(parent_path + "/dataset/" + dataset_name + "/optuna_results/" + best_param_file, "r") as fr:
            best_params_dict[generator_name] = json.load(fr)
    else:
        dict_params_dp = {}
        for f in os.listdir(parent_path + "/dataset/" + dataset_name + "/optuna_results"):
            if (f.endswith(generator_name + '.json') & (name_config_1 in f) & (name_config_2 in f)):
                for eps in [1, 3, 5, 7, 10]:
                    best_param_file = ""
                    if (f"DP_eps_{eps}" in f):
                        best_param_file = f
                        with open(parent_path + "/dataset/" + dataset_name + "/optuna_results/" + best_param_file, "r") as fr:
                            dict_params_dp[eps] = json.load(fr)
        best_params_dict[generator_name] = dict_params_dp


In [ ]:
import surv_hivae # our model

n_generated_dataset = 200

dict_eps_generated_data = {}
for eps in [1.0, 3.0, 5.0, 7.0, 10.0]:
    data_gen_control_eps = surv_hivae.run(df_init_control_encoded,
                                            miss_mask_control, 
                                            true_miss_mask_control,
                                            feat_types_dict,
                                            n_generated_dataset,
                                            seed=1,
                                            apply_rounding=True,
                                            params=best_params_dict['HI-VAE_weibull_DP'][round(eps)],
                                            differential_privacy=True,
                                            target_epsilon=eps)
    list_df_gen_control = []
    for j in range(n_generated_dataset):
        df_gen_control_j = pd.DataFrame(data_gen_control_eps[j].numpy(), columns=control_fnames)
        df_gen_control_j['treatment'] = 0
        list_df_gen_control.append(df_gen_control_j)
    list_df_gen_control_recoded = [
        data_processing.decode_categoricals(df, dict_cat_map_control)
        for df in list_df_gen_control]
    dict_eps_generated_data[eps] = list_df_gen_control_recoded


In [ ]:
# Non DP version
data_gen_control = surv_hivae.run(df_init_control_encoded,
                                  miss_mask_control, 
                                  true_miss_mask_control,
                                  feat_types_dict,
                                  n_generated_dataset,
                                  seed=1,
                                  apply_rounding=True,
                                  params=best_params_dict['HI-VAE_weibull'],
                                  differential_privacy=False)
list_df_gen_control = []
for j in range(n_generated_dataset):
    df_gen_control_j = pd.DataFrame(data_gen_control_eps[j].numpy(), columns=control_fnames)
    df_gen_control_j['treatment'] = 0
    list_df_gen_control.append(df_gen_control_j)
list_df_gen_control_recoded = [
    data_processing.decode_categoricals(df, dict_cat_map_control)
    for df in list_df_gen_control]
dict_eps_generated_data['No privacy'] = list_df_gen_control_recoded

### 2.2. 67% of the dataset

In [ ]:
import json
name_config_1 = f"best_params_traincontrol_{dataset_name}_aug_Ncontrol2%3"
name_config_2 = "ntrials150_DetectXGB"

generators_sel = ["HI-VAE_weibull", "HI-VAE_piecewise", "HI-VAE_weibull_DP", "HI-VAE_piecewise_DP"]

best_params_dict = {}
for generator_name in generators_sel:
    best_param_file = ""
    if "DP" not in generator_name:
        # best_param_file = [item for item in best_param_files if generator_name in item][0]
        for f in os.listdir(parent_path + "/dataset/" + dataset_name + "/optuna_results"):
            if (f.endswith(generator_name + '.json') & (name_config_1 in f) & (name_config_2 in f)):
                best_param_file = f
        with open(parent_path + "/dataset/" + dataset_name + "/optuna_results/" + best_param_file, "r") as fr:
            best_params_dict[generator_name] = json.load(fr)
    else:
        dict_params_dp = {}
        for f in os.listdir(parent_path + "/dataset/" + dataset_name + "/optuna_results"):
            if (f.endswith(generator_name + '.json') & (name_config_1 in f) & (name_config_2 in f)):
                for eps in [1, 3, 5, 7, 10]:
                    best_param_file = ""
                    if (f"DP_eps_{eps}" in f):
                        best_param_file = f
                        with open(parent_path + "/dataset/" + dataset_name + "/optuna_results/" + best_param_file, "r") as fr:
                            dict_params_dp[eps] = json.load(fr)
        best_params_dict[generator_name] = dict_params_dp


In [ ]:
# In order to run a membership inference attack on real data, we need to decompose the initial set in a train vs holdout set.
# We are therefore forced to exclude some rows from the training. We choose to train using 2/3 of the original data.

list_indexes_train = sorted(np.random.choice(len(df_init_control), size=int(2/3 * len(df_init_control)), replace=False))

df_init_control_train = df_init_control[df_init_control.index.isin(list_indexes_train)].reset_index(drop=True)
df_init_control_holdout = df_init_control[~df_init_control.index.isin(list_indexes_train)].reset_index(drop=True)

df_init_control_encoded_train = df_init_control_encoded[df_init_control_encoded.index.isin(list_indexes_train)].reset_index(drop=True)

In [ ]:
import surv_hivae # our model

# In this case, we generate only a number of rows equal to the number of rows of the training set (2/3 of the  initial data), since
# since we will only use the synthetic data to run the GANLeaks attack and evaluate privacy.

dict_eps_generated_data_aug = {}
for eps in [1.0, 3.0, 5.0, 7.0, 10.0]:
    data_gen_control_eps = surv_hivae.run(df_init_control_encoded_train,
                                            miss_mask_control[list_indexes_train], 
                                            true_miss_mask_control[list_indexes_train],
                                            feat_types_dict,
                                            n_generated_dataset,
                                            seed=1,
                                            apply_rounding=True,
                                            params=best_params_dict['HI-VAE_weibull_DP'][round(eps)],
                                            differential_privacy=True,
                                            target_epsilon=eps)
    list_df_gen_control = []
    for j in range(n_generated_dataset):
        df_gen_control_j = pd.DataFrame(data_gen_control_eps[j].numpy(), columns=control_fnames)
        df_gen_control_j['treatment'] = 0
        list_df_gen_control.append(df_gen_control_j)
    list_df_gen_control_recoded = [
        data_processing.decode_categoricals(df, dict_cat_map_control)
        for df in list_df_gen_control]
    dict_eps_generated_data_aug[eps] = list_df_gen_control_recoded


In [ ]:
# Non DP version
data_gen_control = surv_hivae.run(df_init_control_encoded_train,
                                  miss_mask_control[list_indexes_train], 
                                  true_miss_mask_control[list_indexes_train],
                                  feat_types_dict,
                                  n_generated_dataset,
                                  seed=1,
                                  apply_rounding=True,
                                  params=best_params_dict['HI-VAE_weibull'],
                                  differential_privacy=False)
list_df_gen_control = []
for j in range(n_generated_dataset):
    df_gen_control_j = pd.DataFrame(data_gen_control_eps[j].numpy(), columns=control_fnames)
    df_gen_control_j['treatment'] = 0
    list_df_gen_control.append(df_gen_control_j)
dict_eps_generated_data_aug['No privacy'] = list_df_gen_control

## 3. Model evaluation

### 3.1. Metrics computed per generated dataset

In [ ]:
from metrics import general_metrics_modular


simple_score_metrics = {'stats': ['jensenshannon_dist', 'survival_km_distance'],
                        'detection': ['detection_xgb'],
                        'privacy': ['k-map', 'identifiability_score']}

list_score_df_eps = []
for eps in [1.0, 3.0, 5.0, 7.0, 10.0, 'No privacy']:
    try:
        score_df_eps = general_metrics_modular(data_processing.decode_categoricals(df_init_control, dict_cat_map_control), dict_eps_generated_data[eps], 
                                                f"hivae_eps_{round(eps)}", simple_score_metrics, include_nndr=True, include_tableone_min_p_value=True,
                                                categorical=categorical, continuous=continuous,
                                                nonnormal=continuous)
        score_df_eps['eps'] = round(eps)
    except:
        score_df_eps = general_metrics_modular(data_processing.decode_categoricals(df_init_control, dict_cat_map_control), dict_eps_generated_data[eps], 
                                                "hivae_no_privacy", simple_score_metrics, include_nndr=True, include_tableone_min_p_value=True,
                                                categorical=categorical, continuous=continuous,
                                                nonnormal=continuous)
        score_df_eps['eps'] = eps
    list_score_df_eps.append(score_df_eps)
score_df_complete = pd.concat(list_score_df_eps)

### 3.2. Membership Inference Attack 

In [ ]:
from metrics import membership_inference_attack
list_mia_scores_eps = []
for eps in [1.0, 3.0, 5.0, 7.0, 10.0, 'No privacy']:
    mia_results_dict = membership_inference_attack(df_init_control_train, df_init_control_holdout, dict_eps_generated_data_aug[eps],
                                                categorical=categorical+['censor', 'treatment'], bootstrap=True, return_boot_auc=True)
    mia_score_df = pd.DataFrame()
    mia_score_df['AUC-ROC'] = mia_results_dict['boot_auc']
    try:
        mia_score_df['generator'] = f'hivae_eps_{round(eps)}'
    except:
        mia_score_df['generator'] = 'hivae_eps_eps'
    list_mia_scores_eps.append(mia_score_df)
mia_scores_df = pd.concat(list_mia_scores_eps, ignore_index=True)

## 4. Visualization

In [ ]:
from visualization import visualize_perf_vs_privacy

dict_metrics_fidelity = {"Fidelity": [["J-S distance", "min"], ["Detection XGB", 0.5]]}
dict_metrics_utility = {"Utility": [["Survival curves distance", "min"], ["TableOne min p-value", "max"]]}
dict_metrics_privacy = {"Privacy": [["K-map score", "max"], ["Identifiability score", "min"], ["NNDR", "max"]]}

fig = visualize_perf_vs_privacy(score_df_complete, dict_metrics_fidelity, ncols=2,
                          suptitle="Fidelity metrics across privacy levels")
fig = visualize_perf_vs_privacy(score_df_complete, dict_metrics_utility, ncols=2,
                          suptitle="Utility metrics across privacy levels")
fig = visualize_perf_vs_privacy(score_df_complete, dict_metrics_privacy, ncols=3,
                          suptitle="Privacy metrics across privacy levels")

In [ ]:
fig = visualize_perf_vs_privacy(mia_scores_df, {"Privacy": [["AUC-ROC", 0.5]]}, ncols=1,
                                suptitle="MIA-based metric across privacy levels")